In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ── CELL 2: Imports ────────────────────────────────────
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import ASTForAudioClassification
warnings.filterwarnings('ignore')
print('Imports done!')

2026-03-16 00:42:45.703832: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773621766.053353      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773621766.147459      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773621766.962261      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773621766.962324      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773621766.962327      24 computation_placer.cc:177] computation placer alr

Imports done!


In [3]:
# ── CELL 3: CONFIG ─────────────────────────────────────
# 🔧 The only things you should need to change:

# Path to your trained checkpoint (.pt file)
MODEL_PT_PATH = '/kaggle/input/datasets/smrkamboj/best-ast-model/best_model_phase2.pth'

SAMPLE_RATE  = 16000
DURATION     = 20                  # must match training
MAX_LENGTH   = SAMPLE_RATE * DURATION

BATCH_SIZE   = 64                  # inference only — no gradients, use larger batch
NUM_WORKERS  = 4
PREFETCH     = 4

N_TTA        = 7                   # TTA crops per test file (try 7 for better score)

# SpecAugment params — must match training (used only to build GPUAudioTransform,
# augment=False at inference so these values don't affect predictions)
TIME_MASK_PARAM = 192
FREQ_MASK_PARAM = 48

BASE_PATH = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'

GENRES   = ['blues','classical','country','disco','hiphop',
            'jazz','metal','pop','reggae','rock']
label2id = {g: i for i, g in enumerate(GENRES)}
id2label = {i: g for g, i in label2id.items()}
NUM_LABELS = len(GENRES)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
print(f'N_TTA   : {N_TTA}')
print(f'Batch   : {BATCH_SIZE}')
print(f'Workers : {NUM_WORKERS}')

Device  : cuda
N_TTA   : 7
Batch   : 64
Workers : 4


In [4]:
# ── CELL 4: Audio Helper Functions ─────────────────────

def load_audio(path):
    """Load mono audio at SAMPLE_RATE using torchaudio."""
    waveform, sr = torchaudio.load(path)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    if waveform.shape[0] > 1:        # stereo → mono
        waveform = waveform.mean(0, keepdim=True)
    return waveform.squeeze(0).numpy().astype(np.float32)

def normalize(audio):
    return audio / (np.max(np.abs(audio)) + 1e-6)

def crop_random(audio):
    """Random crop — each TTA pass picks a different segment."""
    if len(audio) >= MAX_LENGTH:
        start = random.randint(0, len(audio) - MAX_LENGTH)
        return audio[start : start + MAX_LENGTH]
    return np.pad(audio, (0, MAX_LENGTH - len(audio)))

print('Audio helpers ready!')

Audio helpers ready!


In [5]:
# ── CELL 5: GPU Audio Transform ────────────────────────
# Converts a batch of raw waveforms → AST-compatible log-mel spectrograms
# entirely on GPU. Must match the transform used during training.

class GPUAudioTransform(nn.Module):
    AST_MEAN        = -4.2677393
    AST_STD         =  4.5689974
    AST_TIME_FRAMES = 1024

    def __init__(
        self,
        sample_rate     = SAMPLE_RATE,
        n_mels          = 128,
        n_fft           = 400,
        hop_length      = 313,     # ceil(320000/1023): 20s → 1022 frames, padded to 1024
        f_min           = 50.0,
        f_max           = 8000.0,
        time_mask_param = TIME_MASK_PARAM,
        freq_mask_param = FREQ_MASK_PARAM,
    ):
        super().__init__()
        self.mel_spec = T.MelSpectrogram(
            sample_rate = sample_rate,
            n_fft       = n_fft,
            hop_length  = hop_length,
            n_mels      = n_mels,
            f_min       = f_min,
            f_max       = f_max,
            power       = 2.0,
            center      = True,
        )
        self.amplitude_to_db = T.AmplitudeToDB(stype='power', top_db=80)
        # SpecAugment layers — defined for API consistency, never used at inference
        self.time_mask_1 = T.TimeMasking(time_mask_param, iid_masks=True)
        self.time_mask_2 = T.TimeMasking(time_mask_param, iid_masks=True)
        self.freq_mask_1 = T.FrequencyMasking(freq_mask_param, iid_masks=True)
        self.freq_mask_2 = T.FrequencyMasking(freq_mask_param, iid_masks=True)

    def forward(self, waveforms: torch.Tensor, augment: bool = False) -> torch.Tensor:
        spec = self.mel_spec(waveforms)                    # (B, n_mels, T_frames)
        spec = self.amplitude_to_db(spec)
        # Pad or truncate to exactly 1024 frames (AST positional embedding size)
        T_frames = spec.shape[2]
        if T_frames > self.AST_TIME_FRAMES:
            spec = spec[:, :, :self.AST_TIME_FRAMES]
        elif T_frames < self.AST_TIME_FRAMES:
            spec = torch.nn.functional.pad(spec, (0, self.AST_TIME_FRAMES - T_frames))
        spec = (spec - self.AST_MEAN) / (self.AST_STD * 2)
        if augment:
            spec = self.time_mask_1(spec)
            spec = self.time_mask_2(spec)
            spec = self.freq_mask_1(spec)
            spec = self.freq_mask_2(spec)
        return spec.transpose(1, 2)                        # (B, 1024, 128)

gpu_transform = GPUAudioTransform().to(DEVICE)
with torch.no_grad():
    dummy = torch.randn(2, MAX_LENGTH).to(DEVICE)
    out   = gpu_transform(dummy, augment=False)
    assert out.shape == (2, 1024, 128), f'Unexpected shape: {out.shape}'
print(f'GPUAudioTransform ready on {DEVICE}  |  output shape: {out.shape}')

GPUAudioTransform ready on cuda  |  output shape: torch.Size([2, 1024, 128])


In [6]:
# ── CELL 6: Load Model from .pt Checkpoint ─────────────
# Builds the AST architecture (10-class head) then loads
# your trained weights — no HuggingFace download needed.

MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels              = NUM_LABELS,
    id2label                = id2label,
    label2id                = label2id,
    ignore_mismatched_sizes = True,   # replaces 527-class head with 10-class
)

# Load your trained weights
state_dict = torch.load(MODEL_PT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded from : {MODEL_PT_PATH}')
print(f'Total parameters  : {total_params:,}')
print(f'Device            : {DEVICE}')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded from : /kaggle/input/datasets/smrkamboj/best-ast-model/best_model_phase2.pth
Total parameters  : 86,196,490
Device            : cuda


In [7]:
# ── CELL 7: Test TTA Dataset ────────────────────────────
# Flattens N_test × N_TTA into one flat Dataset so the
# DataLoader can parallelize I/O across all crops at once.

class TestTTADataset(Dataset):
    """
    Each item is one (file_idx, random_crop) pair.
    Total length = len(test_df) × N_TTA.
    """
    def __init__(self, test_df, base_path, n_tta=N_TTA):
        self.test_df   = test_df
        self.base_path = base_path
        self.n_tta     = n_tta
        self.n_files   = len(test_df)

    def __len__(self):
        return self.n_files * self.n_tta

    def __getitem__(self, idx):
        file_idx = idx % self.n_files
        row      = self.test_df.iloc[file_idx]
        path     = os.path.join(self.base_path, row['filename'])
        audio    = load_audio(path)
        cropped  = crop_random(audio)   # different random crop per TTA pass
        cropped  = normalize(cropped)
        return file_idx, torch.from_numpy(cropped)  # (file_idx, T)

print('TestTTADataset defined!')

TestTTADataset defined!


In [8]:
# ── CELL 8: Generate Submission ────────────────────────

test_df = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
n_files = len(test_df)
print(f'Test files : {n_files}  |  TTA crops : {N_TTA}  |  Total items : {n_files * N_TTA}')

tta_dataset = TestTTADataset(test_df, BASE_PATH, n_tta=N_TTA)
tta_loader  = DataLoader(
    tta_dataset,
    batch_size         = BATCH_SIZE,
    shuffle            = False,
    num_workers        = NUM_WORKERS,
    pin_memory         = True,
    persistent_workers = True,
    prefetch_factor    = PREFETCH,
)

# Accumulate softmax probabilities per file: (n_files, NUM_LABELS)
prob_accum = torch.zeros(n_files, NUM_LABELS, dtype=torch.float32)

with torch.no_grad():
    for file_indices, waveforms in tqdm(tta_loader, desc=f'Inference (TTA ×{N_TTA})'):
        waveforms    = waveforms.to(DEVICE)
        input_values = gpu_transform(waveforms, augment=False)    # (B, 1024, 128)
        outputs      = model(input_values=input_values)
        probs        = torch.softmax(outputs.logits, dim=1).cpu() # (B, NUM_LABELS)
        for i, fid in enumerate(file_indices):
            prob_accum[fid] += probs[i]

# Average over N_TTA crops and take argmax
prob_accum /= N_TTA
pred_ids    = prob_accum.argmax(dim=1).tolist()

all_ids   = test_df['id'].tolist()
all_preds = [id2label[p] for p in pred_ids]

submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print(f'\nSubmission saved!')
print(f'Total predictions : {len(submission)}')
print(f'\nGenre distribution:')
print(submission['genre'].value_counts())
print(f'\nFirst 5 predictions:')
print(submission.head())

Test files : 3020  |  TTA crops : 7  |  Total items : 21140


Inference (TTA ×7): 100%|██████████| 331/331 [15:12<00:00,  2.76s/it]


Submission saved!
Total predictions : 3020

Genre distribution:
genre
pop          452
reggae       403
rock         359
hiphop       344
metal        328
jazz         324
classical    256
disco        229
blues        219
country      106
Name: count, dtype: int64

First 5 predictions:
   id    genre
0   1      pop
1   2     jazz
2   3    disco
3   4    metal
4   5  country
